# Bias Node Research Notebook

Generic notebook for testing bias nodes and generating comprehensive reports.

## How to Use:
1. **Edit the `BIAS_NODES_TO_TEST` list** in the configuration cell below
2. Add/remove any bias nodes you want to test
3. Run all cells to generate:
   - Feature analysis reports (decile plots, distributions, permutation tests)
   - QuantStats HTML tearsheets (Sharpe, Sortino, drawdowns, etc.)

## Output:
- Reports saved to: `research/reports/bias_nodes/{node_name}/`
- Tearsheets saved to: `research/reports/bias_nodes/{node_name}/{node_name}_tearsheet.html`

In [ ]:
import sys
from pathlib import Path

# Get project root
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

# Create reports directory
reports_dir = project_root / 'research' / 'reports' / 'bias_nodes'
reports_dir.mkdir(parents=True, exist_ok=True)
print(f"Reports will be saved to: {reports_dir}")

In [ ]:
import time
from utils.core.enums import Ticker, TimeFrame, Direction
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from feature_extraction.feature_extractor import extract_features_for_bias_node
from feature_selection.base_models import BaseModel, QuantileBinningModel, TwoBinBinningModel
from eda.feature_explorer import FeatureExplorer
from metrics.plotting.graphing.quantstats_reports import generate_tearsheet
import utils.core.helpers as helpers

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

---
## Configuration

**Edit the `BIAS_NODES_TO_TEST` list below to add/remove bias nodes.**

Each entry is a dict with:
- `name`: Human-readable name for reports
- `spec`: Bias node specification (module_name, timeframes, params)
- `is_continuous`: True for continuous features (0-100), False for rule-based (0/1)

In [ ]:
# =============================================================================
# CONFIGURATION - Edit this cell to customize your research
# =============================================================================

# Tickers to test on (ensemble)
ENSEMBLE_TICKERS = [Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY]

# Date range for analysis
START_DATE = datetime(2000, 1, 1)
END_DATE = datetime(2024, 12, 31)

# Permutation test repetitions (higher = more accurate p-values but slower)
PERMUTATION_REPS = 100

# Generate QuantStats tearsheets (set to False to skip)
GENERATE_TEARSHEETS = True

# Strategy: 'long', 'short', or 'long-short'
STRATEGY = 'long'
# Target column: 'log_return', 'log_return_ewsd', 'log_return_ewsd'
TARGET_COL = 'log_return'

# =============================================================================
# CACHING CONFIGURATION (NEW)
# =============================================================================
# Set USE_CACHE = True to use pre-computed cached values (~50-100x faster)
# First run with POPULATE_CACHE = True to create caches, then set to False
USE_CACHE = True
POPULATE_CACHE = False  # Set True on first run to populate caches

# =============================================================================
# BIAS NODES TO TEST - Add/remove entries as needed
# =============================================================================
# Each entry needs:
#   - 'name': Display name for reports
#   - 'spec': {'module_name': ..., 'timeframes': [...], 'params': {...}}
#   - 'is_continuous': True for continuous (use QuantileBinning), False for rule-based (TwoBinBinning)

BIAS_NODES_TO_TEST = [
    # ---------------------------------------------------------------------
    # CONTINUOUS FEATURES (0-100 or similar range) - Use QuantileBinning
    # ---------------------------------------------------------------------


    

]

print(f"Configuration loaded:")
print(f"  Tickers: {[t.name for t in ENSEMBLE_TICKERS]}")
print(f"  Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"  Nodes to test: {len(BIAS_NODES_TO_TEST)}")
print(f"  Generate tearsheets: {GENERATE_TEARSHEETS}")
print(f"  Strategy: {STRATEGY}")
print(f"  Target col: {TARGET_COL}")
print(f"  USE_CACHE: {USE_CACHE}")
print(f"  POPULATE_CACHE: {POPULATE_CACHE}")

In [ ]:
# =============================================================================
# LOAD BIAS NODES FROM VAULT (AUTO)
# =============================================================================
from typing import Any, Dict, List
import json

vault_d_dir = project_root / 'vault' / 'D'
feature_files = sorted(vault_d_dir.glob('*/features/*.json'))

if not feature_files:
    raise FileNotFoundError(f"No vault feature configs found under: {vault_d_dir}")


def _to_timeframes(timeframes: List[Any]) -> List[TimeFrame]:
    return [TimeFrame[tf] if isinstance(tf, str) else tf for tf in timeframes]


def _binning_models(base_models: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [
        {
            'model_id': model.get('model_id'),
            'binning_model_type': model.get('binning_model_type'),
            'strategy': model.get('strategy', model.get('binning_model_params', {}).get('strategy', 'long')),
            'binning_model_params': model.get('binning_model_params', {}),
            'is_fitted': model.get('is_fitted', False),
            'fitted_params': model.get('fitted_params')
        }
        for model in base_models
    ]


def _is_continuous(base_models: List[Dict[str, Any]]) -> bool:
    model_types = [model.get('binning_model_type') for model in base_models]
    return all(model_type != 'TwoBinBinningModel' for model_type in model_types if model_type is not None)


VAULT_FEATURE_CONFIGS = [
    {
        'path': feature_path,
        'source_ensemble': feature_path.parent.parent.name,
        'feature_config': json.loads(feature_path.read_text())
    }
    for feature_path in feature_files
]

BIAS_NODES_TO_TEST = [
    {
        'name': cfg['feature_config'].get('feature_name', cfg['feature_config'].get('feature_column', cfg['path'].stem)),
        'spec': {
            **cfg['feature_config']['bias_node_spec'],
            'timeframes': _to_timeframes(
                cfg['feature_config']['bias_node_spec'].get('timeframes', [TimeFrame.D])
            )
        },
        'is_continuous': _is_continuous(cfg['feature_config'].get('base_models', [])),
        'binning_models': _binning_models(cfg['feature_config'].get('base_models', [])),
        'source_ensemble': cfg['source_ensemble'],
        'feature_column': cfg['feature_config'].get('feature_column')
    }
    for cfg in VAULT_FEATURE_CONFIGS
]

print("Vault bias nodes loaded:")
print(f"  Feature configs: {len(VAULT_FEATURE_CONFIGS)}")
print(f"  Nodes to test: {len(BIAS_NODES_TO_TEST)}")

In [ ]:
# =============================================================================
# CACHE POPULATION (Run once to create caches, then set POPULATE_CACHE=False)
# =============================================================================
import time
from itertools import product

if POPULATE_CACHE:
    from utils.cache.cache_manager import CacheManager
    
    print("="*70)
    print("POPULATING CACHES (this takes time but only needs to be done once)")
    print("="*70)
    
    # Point to correct candle data directory
    manager = CacheManager(candle_dir=str(project_root / 'data' / 'ohlc_data'))
    
    # Build list of all bias node specs to cache (expand parameter grids)
    all_specs = []
    for node_config in BIAS_NODES_TO_TEST:
        spec = node_config['spec']
        module_name = spec['module_name']
        timeframes = spec.get('timeframes', [TimeFrame.D])
        params = spec.get('params', {})
        
        # Expand parameter grid
        keys = list(params.keys())
        values = [v if isinstance(v, list) else [v] for v in params.values()]
        if keys:
            param_combos = [dict(zip(keys, combo)) for combo in product(*values)]
        else:
            param_combos = [{}]
        
        for param_combo in param_combos:
            all_specs.append({
                'module_name': module_name,
                'params': param_combo,
                'timeframes': timeframes
            })
    
    # Also need ATR and EWSD for volatility scaling
    all_specs.append({'module_name': 'atr', 'params': {'period': 252}, 'timeframes': [TimeFrame.D]})
    all_specs.append({'module_name': 'ewsd', 'params': {}, 'timeframes': [TimeFrame.D]})
    
    print(f"  Total bias node specs to cache: {len(all_specs)}")
    print(f"  Tickers: {[t.name for t in ENSEMBLE_TICKERS]}")
    print(f"  Total cache files: {len(all_specs) * len(ENSEMBLE_TICKERS)}")
    
    t0 = time.time()
    result = manager.populate_cache(
        bias_node_specs=all_specs,
        tickers=ENSEMBLE_TICKERS,
        start_date=START_DATE,
        end_date=END_DATE,
        show_progress=True,
        overwrite_existing=False  # Skip existing caches
    )
    elapsed = time.time() - t0
    
    print(f"\n{'='*70}")
    print(f"CACHE POPULATION COMPLETE")
    print(f"  Success: {result['success']}/{result['total']}")
    print(f"  Skipped (already cached): {result['skipped']}")
    print(f"  Failed: {result['failed']}")
    print(f"  Time: {elapsed:.1f}s")
    print(f"{'='*70}")
else:
    print("POPULATE_CACHE=False - Skipping cache population")
    print("Set POPULATE_CACHE=True and rerun to create caches")

---
## Helper Functions

In [ ]:
from research.bias_node_helpers import (
    build_feature_metadata,
    generate_node_tearsheet,
    get_best_feature_from_permutation,
    get_binning_model,
    test_bias_node,
)

---
## Run All Tests

This cell iterates through all bias nodes in `BIAS_NODES_TO_TEST` and generates reports.

In [ ]:
# Run tests for all bias nodes
all_results = {}
tearsheet_paths = {}
total_extract_time = 0

print("="*70)
print(f"STARTING BIAS NODE RESEARCH - {len(BIAS_NODES_TO_TEST)} nodes to test")
print(f"Mode: {'CACHED' if USE_CACHE else 'STREAMING'}")
print("="*70)

t_start = time.time()

for i, node_config in enumerate(BIAS_NODES_TO_TEST, 1):
    node_name = node_config['name']
    bias_spec = node_config['spec']
    is_continuous = node_config['is_continuous']
    
    print(f"\n[{i}/{len(BIAS_NODES_TO_TEST)}] Processing {node_name}...")
    
    # Generate feature analysis report
    results = test_bias_node(
        bias_spec=bias_spec,
        node_name=node_name,
        is_continuous=is_continuous,
        tickers=ENSEMBLE_TICKERS,
        start_date=START_DATE,
        end_date=END_DATE,
        reports_dir=reports_dir,
        permutation_reps=PERMUTATION_REPS,
        show_plots=False,
        export=True,
        use_cache=USE_CACHE,
        strategy=STRATEGY,
        target_col=TARGET_COL
    )
    all_results[node_name] = results
    
    if results is not None and 'extract_time' in results:
        total_extract_time += results['extract_time']
    
    # Generate QuantStats tearsheet if enabled
    # Pass pre-extracted features/targets to ensure consistency with feature analysis
    if GENERATE_TEARSHEETS and results is not None:
        perm_df = results.get('permutation_test', None)
        features_df = results.get('features_df', None)
        targets_df = results.get('targets_df', None)
        
        binning_model = results.get('binning_model')
        tearsheet_path = generate_node_tearsheet(
            node_name=node_name,
            is_continuous=is_continuous,
            start_date=START_DATE,
            end_date=END_DATE,
            reports_dir=reports_dir,
            perm_df=perm_df,
            features_df=features_df,
            targets_df=targets_df,
            strategy=STRATEGY,
            target_col=TARGET_COL,
            tickers=ENSEMBLE_TICKERS,
            binning_model=binning_model,
        )
        tearsheet_paths[node_name] = tearsheet_path

t_total = time.time() - t_start

print("\n" + "="*70)
print("ALL TESTS COMPLETE")
print(f"Total time: {t_total:.1f}s")
print(f"Feature extraction time: {total_extract_time:.1f}s")
print(f"Mode: {'CACHED' if USE_CACHE else 'STREAMING'}")
print("="*70)

---
## Summary of Results

In [ ]:
# Compile summary of all results
print("\n" + "="*70)
print("SUMMARY: Bias Node Research Results")
print("="*70)

summary_data = []
for name, result in all_results.items():
    # Check for permutation_test (correct key name)
    if result is not None and 'permutation_test' in result and result['permutation_test'] is not None:
        perm_df = result['permutation_test']
        if len(perm_df) > 0:
            # Get best feature by p-value (key might be 'pval' or 'p_value')
            pval_col = 'pval' if 'pval' in perm_df.columns else 'p_value'
            best_row = perm_df.loc[perm_df[pval_col].idxmin()]
            best_pval = best_row[pval_col]
            summary_data.append({
                'Node': name,
                'Best Feature': best_row['feature'][:40] + '...' if len(best_row['feature']) > 40 else best_row['feature'],
                'P-Value': best_pval,
                'Significant': 'YES' if best_pval <= 0.1 else 'NO',
                'Status': 'OK'
            })
        else:
            summary_data.append({'Node': name, 'Status': 'No permutation results'})
    else:
        summary_data.append({'Node': name, 'Status': 'FAILED' if result is None else 'No results'})

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

# Count significant features
if 'Significant' in summary_df.columns:
    n_significant = (summary_df['Significant'] == 'YES').sum()
    n_total = len(summary_df[summary_df['Status'] == 'OK'])
    print(f"\n{n_significant}/{n_total} nodes have statistically significant features (p <= 0.1)")

print(f"\nAll reports saved to: {reports_dir}")

In [ ]:
# List all generated report files
import os

print("\nGenerated Report Files:")
print("="*70)

for root, dirs, files in os.walk(reports_dir):
    level = root.replace(str(reports_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    folder_name = os.path.basename(root)
    if folder_name:
        print(f"{indent}[DIR] {folder_name}/")
    for file in files:
        print(f"{indent}  - {file}")

---
## QuantStats Tearsheet Links

Open these HTML files in your browser to view detailed performance metrics.

In [ ]:
if GENERATE_TEARSHEETS and tearsheet_paths:
    print("\nQuantStats Tearsheets Generated:")
    print("="*70)
    for node_name, path in tearsheet_paths.items():
        if path:
            print(f"  {node_name}: {path}")
else:
    print("No tearsheets generated (GENERATE_TEARSHEETS=False or all failed)")

In [ ]:
# =============================================================================
# VAULT RE-ORG: DELETE + RESAVE
# =============================================================================
import shutil
from typing import Any, Dict, Tuple
import numpy as np

from ensemble.vault_manager import create_ensemble_directory
from feature_selection.base_models import (
    BaseModel,
    DecisionTreeBinningModel,
    QuantileBinningModel,
    TwoBinBinningModel
)
from utils.core.enums import Direction, TimeFrame

if 'VAULT_FEATURE_CONFIGS' not in globals():
    raise ValueError("Run the vault load cell first.")

vault_d_dir = project_root / 'vault' / 'D'
vault_d_dir.mkdir(parents=True, exist_ok=True)

existing_ensembles = [path for path in vault_d_dir.iterdir() if path.is_dir()]
_ = [shutil.rmtree(path) for path in existing_ensembles]


def _normalize_bias_spec(bias_node_spec: Dict[str, Any]) -> Dict[str, Any]:
    return {
        **bias_node_spec,
        'timeframes': _to_timeframes(bias_node_spec.get('timeframes', [TimeFrame.D]))
    }


def _target_ensemble(bias_spec: Dict[str, Any]) -> Tuple[str, Direction]:
    module_name = bias_spec.get('module_name', '')
    params = bias_spec.get('params', {})
    regime_filter = params.get('regime_filter', [])
    regime_values = regime_filter if isinstance(regime_filter, list) else [regime_filter]

    if module_name == 'buy_hold':
        return ('buy-hold_indices', Direction.LONG)
    if module_name in {'cumulative_rsi', 'rsi', 'ultimate_c'}:
        return ('mean-reversion_indices', Direction.LONG)
    if module_name == 'rsi_regime':
        return (
            'mean-reversion_indices',
            Direction.SHORT if 'bearish' in regime_values else Direction.LONG
        )
    if module_name == 'ewmac':
        return ('momentum_indices', Direction.LONG)

    raise ValueError(f"No ensemble mapping for module_name='{module_name}'")


def _build_binning_model(model_config: Dict[str, Any]) -> Any:
    model_type = model_config.get('binning_model_type', 'QuantileBinningModel')
    binning_params = dict(model_config.get('binning_model_params', {}))
    strategy = model_config.get('strategy', binning_params.get('strategy', 'long'))
    binning_params.pop('strategy', None)
    binning_params.pop('normalize_by', None)

    if model_type == 'TwoBinBinningModel':
        binning_params.pop('n_bins', None)
        model = TwoBinBinningModel(**binning_params, strategy=strategy)
    elif model_type == 'DecisionTreeBinningModel':
        model = DecisionTreeBinningModel(**binning_params, strategy=strategy)
    else:
        model = QuantileBinningModel(**binning_params, strategy=strategy)

    fitted_params = model_config.get('fitted_params')
    if model_config.get('is_fitted') and fitted_params:
        thresholds = fitted_params.get('thresholds')
        if thresholds is None or len(thresholds) == 0:
            model.thresholds_ = np.array([])
        else:
            model.thresholds_ = np.array(thresholds)
        model.best_long_bin_ = fitted_params.get('best_long_bin')
        model.best_short_bin_ = fitted_params.get('best_short_bin')
        model.bin_stats_ = fitted_params.get('bin_stats')
        model.is_fitted_ = True

    return model


def _save_model(feature_config: Dict[str, Any], model_config: Dict[str, Any], ensemble_dir: str) -> str:
    bias_spec = _normalize_bias_spec(feature_config['bias_node_spec'])
    binning_model = _build_binning_model(model_config)
    base_model = BaseModel(
        feature_config={'bias_node_spec': bias_spec},
        binning_model=binning_model,
        tickers=ENSEMBLE_TICKERS,
        use_cache=USE_CACHE
    )
    base_model.feature_column = feature_config.get('feature_column')
    return base_model.save_to_vault(ensemble_dir=ensemble_dir, tickers=ENSEMBLE_TICKERS)


normalized_entries = [
    {
        'feature_config': cfg['feature_config'],
        'bias_spec': _normalize_bias_spec(cfg['feature_config']['bias_node_spec'])
    }
    for cfg in VAULT_FEATURE_CONFIGS
]

ensemble_keys = sorted({_target_ensemble(entry['bias_spec']) for entry in normalized_entries})
ensemble_dirs = {
    key: create_ensemble_directory(TimeFrame.D, key[0], key[1], tickers=ENSEMBLE_TICKERS)
    for key in ensemble_keys
}

saved_model_ids = [
    _save_model(
        entry['feature_config'],
        model_cfg,
        ensemble_dirs[_target_ensemble(entry['bias_spec'])]
    )
    for entry in normalized_entries
    for model_cfg in entry['feature_config'].get('base_models', [])
]

print("Vault re-org complete:")
print(f"  Ensembles created: {len(ensemble_dirs)}")
print(f"  Models saved: {len(saved_model_ids)}")
print(f"  Ensemble dirs: {list(ensemble_dirs.values())}")